# Assignment 2 (Coding Part), Problem 1: Hidden Markov Models

Aryan is a fervent ice cream enthusiast who indulges more as the temperature drops. Over $n$ days we have a sequence of temperatures $Y_1,\dots,Y_n$ and the corresponding ice cream consumption $X_1,\dots,X_n$.

The daily ice cream intake $X_t$ takes values 0, 1, or 2 servings. The temperature $Y_t$ takes values 0 (cold), 1 (warm), or 2 (hot).

This is modeled as a Hidden Markov Model, where $Y$ is the hidden state (temperature) and $X$ is the observation (ice cream consumption). Two assumptions hold:

$$P(y_i \mid y_1,\dots,y_{i-1}) = P(y_i \mid y_{i-1})$$
$$P(x_i \mid y_1,\dots,y_i) = P(x_i \mid y_i)$$

```
Y1 --> Y2 --> ... --> Yn
|      |               |
v      v               v
X1     X2              Xn
```

The distribution $P(y_{i+1} \mid y_i)$ is the transition probability, $P(x_i \mid y_i)$ is the emission probability, and $P(y_1)$ is the initial state distribution.

The `HMM` class below stores the initial state distribution, the transition probabilities and the emission probabilities. For instance, `transition_prob[y1][y2]` fetches $P(Y_{i+1}=y_2 \mid Y_i=y_1)$, so the first index is the current state and the second index is the next state. Each row sums to 1. Similarly, `emission_prob[y][x]` fetches $P(X_i=x \mid Y_i=y)$.

Suppose we observe the ice cream counts $x_1,\dots,x_n$. The aim is to find the temperature sequence $y_1,\dots,y_n$ that maximizes $P(y_1,\dots,y_n \mid x_1,\dots,x_n)$.

**(a)** For a chosen $x_1,\dots,x_n$ and $y_1,\dots,y_n$, how do you compute the joint distribution $P(y_1,\dots,y_n,x_1,\dots,x_n)$? *(Write your answer in the text cell below.)*

**(b)** Implement the function `joint_prob` to calculate $P(y_1,\dots,y_n,x_1,\dots,x_n)$. The input `outputs` is a list of the $n$ observed ice cream consumptions and `hiddens` is a list of the hidden temperature states. Type annotations are only guidelines; Python does not check them.

**(c)** Complete `get_most_likely_hidden_states`, which takes an HMM object and the observed consumptions. For every possible temperature sequence (such as $[0,0,0,0,0]$, $[0,0,0,0,1]$, $\dots$), compute $P(y_1,\dots,y_n,x_1,\dots,x_n)$. Then find

$$P(y_1,\dots,y_n \mid x_1,\dots,x_n) = \frac{P(y_1,\dots,y_n,x_1,\dots,x_n)}{P(x_1,\dots,x_n)},$$

where $P(x_1,\dots,x_n)$ is obtained by summing the joint over all possible $y_1,\dots,y_n$. Sort by the conditional probability and output the 10 most likely sequences along with their joint probabilities. A Python list cannot be a dictionary key, so convert it to a tuple first with `tuple()`.

In [25]:
import math
outputs = [0, 0, 2, 2, 1] # X_1 .. X_N
# transition prob..row = today col = tomorrow
transition_prob = [[0.55, 0.25, 0.2],\
                   [0.15, 0.7, 0.15],\
                   [0.2, 0.4, 0.4]]
# how much ice cream eaten given the weather..row = weather col = ice cream eaten
emission_prob = [[0.1, 0.3, 0.6],\
                 [0.2, 0.5, 0.3],\
                 [0.6, 0.3, 0.1]]
start_prob = [0.3, 0.3, 0.4]

### Answer to (a)

We start with the probability for the first days weather and then multiply it with the probabilty of tht days ice cream count. Then for all other days, we multiply the probability of the the weatehrs transition and the probability of the observerd ice cream count given that days weather (the conditional probability)

Formally, using the HMM assumptions the Joint probablity is 

$$
P(y_1,\ldots,y_n,x_1,\ldots,x_n)
=
P(y_1)\,P(x_1\mid y_1)
\prod_{t=2}^{n}
P(y_t\mid y_{t-1})\,P(x_t\mid y_t).
$$

In [26]:
import math

class HMM:
  def __init__(self, transition_prob: list[list[float]], emission_prob: list[list[float]], start: list[float]):
    # transition_prob[i][j] = P(Y_(t+1) = j | Y_t = i)  (row = current state, each row sums to 1)
    self.transition_prob = transition_prob

    # emission_prob[y][x] = P(X_t = x | Y_t = y)
    self.emission_prob = emission_prob

    # start[y] = P(Y_1 = y)
    self.start = start

  def joint_prob(self, outputs: list[int], hiddens: list[int]) -> float:
    if len(outputs) != len(hiddens):
        raise ValueError("outputs and hiddens must have the same length.")

    if len(outputs) == 0:
        return 1.0

    probability = self.start[hiddens[0]]
    probability *= self.emission_prob[hiddens[0]][outputs[0]]

    for day in range(1, len(outputs)):
        previous_weather = hiddens[day - 1]
        current_weather = hiddens[day]
        ice_cream_count = outputs[day]

        probability *= self.transition_prob[previous_weather][current_weather]
        probability *= self.emission_prob[current_weather][ice_cream_count]

    return probability

In [27]:
hmm = HMM(transition_prob, emission_prob, start_prob)

In [28]:
# Test cases
assert math.isclose(hmm.joint_prob([0,0,2,2,1], [1,1,1,1,1]), 0.000129654)
assert math.isclose(hmm.joint_prob([0,0,2,2,1], [2,2,0,0,1]), 0.00028512)
print("joint_prob tests passed.")

joint_prob tests passed.


In [29]:
import itertools

def get_most_likely_hidden_states(hmm: HMM, outputs: list[int]):
    joint_probabilities = {}

    for hiddens in itertools.product([0, 1, 2], repeat=len(outputs)):
        joint_probabilities[hiddens] = hmm.joint_prob(outputs, hiddens)

    total = sum(joint_probabilities.values())

    if total == 0:
        raise ValueError("The observed sequence has zero probability under this HMM.")

    conditional_probabilities = {}

    for hiddens, joint_probability in joint_probabilities.items():
        conditional_probabilities[hiddens] = joint_probability / total

    ranked_sequences = sorted(
        conditional_probabilities,
        key=conditional_probabilities.get,
        reverse=True,
    )

    results = []

    for hiddens in ranked_sequences[:10]:
        results.append((hiddens, joint_probabilities[hiddens]))

    return results


In [30]:
get_most_likely_hidden_states(hmm, outputs)

[((2, 2, 1, 1, 1), 0.0005080320000000001),
 ((2, 2, 0, 0, 0), 0.00037635840000000006),
 ((2, 1, 1, 1, 1), 0.00029635199999999994),
 ((2, 2, 0, 0, 1), 0.00028512000000000003),
 ((2, 2, 0, 1, 1), 0.00018144),
 ((2, 2, 0, 0, 2), 0.0001368576),
 ((1, 1, 1, 1, 1), 0.00012965399999999993),
 ((2, 2, 1, 0, 0), 0.0001026432),
 ((2, 2, 2, 1, 1), 9.676799999999999e-05),
 ((2, 1, 0, 0, 0), 9.408960000000001e-05)]